# Coding attention mechanisms

## Attending to different parts of the input with self-attention

## A simple self-attention mechanism without trainable weights

In [2]:
import torch
#  assuming each row in the tensor is embedding vector for the word or token 
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)



In [4]:
input_query=inputs[1]
input_query

tensor([0.5500, 0.8700, 0.6600])

In [5]:
input_1= inputs[0]
input_1

tensor([0.4300, 0.1500, 0.8900])

In [9]:
#  trying manual 
0.55*0.43+0.87*0.15+0.66*0.89

0.9544

In [10]:
#  same using pytorch
torch.dot(input_query,input_1)

tensor(0.9544)

In [19]:
# res=0.
i=2
res=torch.dot(inputs[i],input_query)
    
res

tensor(1.4754)

In [21]:
torch.empty(inputs.shape[0])

tensor([0., 0., 0., 0., 0., 0.])

In [25]:
#  manully calculating attention scores then we later go fro attention wieghts 

query = inputs[1]  # 2nd input token is the query

attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query) # dot product (transpose not necessary here since they are 1-dim vectors)

print(attn_scores_2)
#  we normalize the socres so we do not end up having large nums 

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [26]:
#  trynna normalize
attn_weights_2_tmp=attn_scores_2/attn_scores_2.sum()
attn_weights_2_tmp

tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])

In [27]:
attn_weights_2_tmp.sum()

tensor(1.0000)

In [30]:
#  a one more to normalize right now we use our way of soft max a simple one 
def softmax_naive(x):
    return torch.exp(x)/torch.exp(x).sum(dim=0)
softmax_naive(attn_scores_2)

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])

In [35]:
#  recommnded use soft max  from pytorch as it is more stable 
attn_weights_2= torch.softmax(attn_scores_2,dim=0)

In [41]:
#  computing context vector
#  for each input we mulpiply each input wth their weight and then we sum 


query = inputs[1] # 2nd input token is the query
#  here first attn wt multipmy with first vector and so on

context_vec_2 = torch.zeros(query.shape)
for i,x_i in enumerate(inputs):
    # print(f"{attn_weights_2[i]} ----> {inputs[i]}")
    context_vec_2 += attn_weights_2[i]*inputs[i]
print(context_vec_2)

#  all this is done w.r.t second input element or token  till here 

tensor([0.4419, 0.6515, 0.5683])


In [42]:

for i,x_i in enumerate(inputs):
    print(i,x_i)

0 tensor([0.4300, 0.1500, 0.8900])
1 tensor([0.5500, 0.8700, 0.6600])
2 tensor([0.5700, 0.8500, 0.6400])
3 tensor([0.2200, 0.5800, 0.3300])
4 tensor([0.7700, 0.2500, 0.1000])
5 tensor([0.0500, 0.8000, 0.5500])


## Simple self-attention mechanism  without trainable weights

In [44]:
#  computin attn wt for all input tokens



attn_scores= torch.empty(6,6)


for i, x_i in enumerate(inputs):
    for j,x_j in enumerate(inputs):
        attn_scores[i,j] = torch.dot(x_i, x_j) 

print(attn_scores)

#  not normalised 

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [45]:
#  doing same as above using mat mul as it is faster than loops
attn_scores=inputs @ inputs.T
attn_scores

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])

In [46]:
#  normailsig to convert scores to wt  so value do not end uup large after summation ,

attn_weights=torch.softmax(attn_scores,dim=1)
attn_weights

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])

In [47]:
#  so in summary we did this to get attn weight for all tokesn
attn_scores=inputs @ inputs.T # matrix multiplication to get  attention scores 
attn_weights= torch.softmax(attn_scores,dim=1)
print(attn_weights)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


In [48]:
#  now the part of context vector

all_context_vecs=attn_weights @ inputs
all_context_vecs

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])

In [50]:

# so In summary basicaly below is all we need to compute context vector

attn_scores=inputs @ inputs.T
attn_weights= torch.softmax(attn_scores,dim=1)
all_context_vecs= attn_weights @ inputs
all_context_vecs

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])

# Now Implementing self-attention with trainable weights

## Computing the attention weights step by step 

In [ ]:
#  till now we were without trainble weights 

